In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import joblib

# STEP 2: LOAD & EXPLORE DATA
print("STEP 2: Loading Data")
df = pd.read_csv('kdd_train.csv')

print("Null values found:\n", df.isnull().sum().sum())
print("\nOriginal Label Distribution:\n", df['labels'].value_counts())


STEP 2: Loading Data
Null values found:
 0

Original Label Distribution:
 labels
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64


In [2]:
import pandas as pd

df = pd.read_csv('kdd_train.csv')
sample_df = df.sample(n=500, random_state=42)
sample_df.to_csv('sample_dataset.csv', index=False)

print("Sample dataset created successfully!")

Sample dataset created successfully!


In [3]:

# STEP 3: CLEAN, NORMALIZE & FEATURE SELECT
print("\nSTEP 3: Cleaning & Normalizing")
df = df.dropna()
df['labels'] = df['labels'].apply(lambda x: 0 if x == 'normal' else 1)
print("Encoded Label Distribution (0=Benign, 1=Attack):\n", df['labels'].value_counts())

numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('labels', errors='ignore')
X_full = df[numeric_cols]

y_full = df['labels'].astype(int)

# 1. SPLIT DATA
X_train_full, X_test_full, y_train, y_test = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

# 2. NORMALIZE
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_full)

# 3. FEATURE SELECTION
print("\nPerforming Feature Selection with SelectFromModel...")
selector = SelectFromModel(RandomForestClassifier(n_estimators=50, random_state=42), threshold='mean')

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

print(f"X_train shape: {X_train_selected.shape}")
print(f"y_train shape: {y_train.shape}")

selected_feature_mask = selector.get_support()
selected_features = numeric_cols[selected_feature_mask]
print(f"Selected {len(selected_features)} highly predictive features out of {len(numeric_cols)}")



STEP 3: Cleaning & Normalizing
Encoded Label Distribution (0=Benign, 1=Attack):
 labels
0    67343
1    58630
Name: count, dtype: int64

Performing Feature Selection with SelectFromModel...
X_train shape: (88181, 12)
y_train shape: (88181,)
Selected 12 highly predictive features out of 38


In [4]:

# STEP 4A: TRAIN RANDOM FOREST (Supervised)
print("\nTraining Random Forest")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train_selected, y_train)

rf_preds = rf_model.predict(X_test_selected)
print("Random Forest F1-Score:", f1_score(y_test, rf_preds))

cm_rf = confusion_matrix(y_test, rf_preds)
print("Random Forest Confusion Matrix:\n", cm_rf)

tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()
fpr_rf = fp_rf / (fp_rf + tn_rf)
print(f"Random Forest False Positive Rate (FPR): {fpr_rf:.4f}")




Training Random Forest
Random Forest F1-Score: 0.9971464896171776
Random Forest Confusion Matrix:
 [[20044    39]
 [   62 17647]]
Random Forest False Positive Rate (FPR): 0.0019


In [5]:
# STEP 4B: TRAIN ISOLATION FOREST (Unsupervised)
print("\nTraining Isolation Forest")
X_train_benign = X_train_selected[y_train == 0]

iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_train_benign)

iso_raw_preds = iso_forest.predict(X_test_selected)
iso_preds_mapped = [0 if p == 1 else 1 for p in iso_raw_preds]

print("Isolation Forest F1-Score:", f1_score(y_test, iso_preds_mapped))
cm_iso = confusion_matrix(y_test, iso_preds_mapped)
print("Isolation Forest Confusion Matrix:\n", cm_iso)

tn_iso, fp_iso, fn_iso, tp_iso = cm_iso.ravel()
fpr_iso = fp_iso / (fp_iso + tn_iso)
print(f"Isolation Forest False Positive Rate (FPR): {fpr_iso:.4f}")


Training Isolation Forest
Isolation Forest F1-Score: 0.8566287474457898
Isolation Forest Confusion Matrix:
 [[19047  1036]
 [ 3665 14044]]
Isolation Forest False Positive Rate (FPR): 0.0516


In [6]:
print("\nModel Comparison Explanation")
print("The Random Forest model achieves a higher F1-score and lower FPR because it is a supervised model trained directly on labeled attack data. Conversely, the Isolation Forest is an unsupervised model trained solely on benign traffic. While the Isolation Forest generates more false positives (alert fatigue), it is theoretically better at catching unseen 'zero-day' attacks by flagging abnormal behavior.")


Model Comparison Explanation
The Random Forest model achieves a higher F1-score and lower FPR because it is a supervised model trained directly on labeled attack data. Conversely, the Isolation Forest is an unsupervised model trained solely on benign traffic. While the Isolation Forest generates more false positives (alert fatigue), it is theoretically better at catching unseen 'zero-day' attacks by flagging abnormal behavior.


In [7]:
import joblib
import numpy as np

# STEP 5: SAVE TRAINED MODELS & PREDICT
print("\nSTEP 5: Saving Models & Script")

joblib.dump(rf_model, 'random_forest_nids.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(iso_forest, 'isolation_forest.pkl')
joblib.dump(selected_features, 'selected_features.pkl')
print("Models, scaler, and features saved successfully.")

# Prediction Script
def predict_new_connection(new_row_df):
    """Loads artifacts, processes a new network connection, and returns a prediction."""
    loaded_rf = joblib.load('random_forest_nids.pkl')
    loaded_scaler = joblib.load('scaler.pkl')
    loaded_features = joblib.load('selected_features.pkl')

    num_cols = new_row_df.select_dtypes(include=[np.number]).columns.drop('labels', errors='ignore')

    new_row_scaled_full = loaded_scaler.transform(new_row_df[num_cols])

    feature_indices = [list(num_cols).index(f) for f in loaded_features]
    new_row_selected = new_row_scaled_full[:, feature_indices]

    prediction = loaded_rf.predict(new_row_selected)
    return "ATTACK (1)" if prediction == 1 else "BENIGN (0)"

sample_connection = df.iloc[0:1].copy()

result = predict_new_connection(sample_connection)
print(f"\nLive Prediction Test Output: {result}")


STEP 5: Saving Models & Script
Models, scaler, and features saved successfully.

Live Prediction Test Output: BENIGN (0)
